# executorlib with SLURM 
Based on the [cmti and cmmg clusters](https://docs.mpcdf.mpg.de/doc/computing/clusters/systems/Sustainable_Materials.html) hosted at the MPCDF for the MPI for Sustainable Materials.

In [1]:
import executorlib

In [2]:
executorlib.__version__

'0.0.1'

In [3]:
executorlib.__path__

['/u/janj/projects/executorlib/src/executorlib']

## Submit Python Function to SLURM
This is just like using `sbatch` to submit shell scripts.

In [4]:
def get_slurm_env():
    import os
    from time import sleep
    sleep(5)
    return {
        k:v for k, v in os.environ.items() if "SLURM_" in k
    }

In [5]:
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env=L
#SBATCH --partition={{partition}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency %}
#SBATCH --dependency=afterok:{{ dependency | join(',') }}
{%- endif %}
{%- if memory_max %}
#SBATCH --mem={{memory_max}}G
{%- endif %}
#SBATCH --ntasks={{cores}}

{{command}}
"""

In [6]:
with executorlib.SlurmClusterExecutor() as exe:
    f1 = exe.submit(
        get_slurm_env,
        resource_dict={
            "submission_template": submission_template, 
            "run_time_max": 180,  # in seconds  
            # "partition": "s.cmfe",
            "partition": "s.cmmg",
            "cores": 1,
            "threads_per_core": 4,
        })
    result = f1.result()
    print(result)

/u/janj/projects/executorlib/src/executorlib/task_scheduler/base.py:156: UserWarning: The following keys are not recognized and cannot be validated: ['partition']
  self._validator(resource_dict=resource_dict)


{'SLURM_MEM_PER_CPU': '1500', 'SLURM_NODEID': '0', 'SLURM_TASK_PID': '272848', 'SLURM_PRIO_PROCESS': '0', 'SLURM_SUBMIT_DIR': '/cmmc/u/janj/notebooks/2026/2026-06-03-prabhath-exe/executorlib_cache/get_slurm_envccfcb90850bb2803c9b37c6ecde017ee', 'SLURM_CELL': 'CMMC', 'SLURM_PROCID': '0', 'SLURM_JOB_GID': '12500', 'SLURM_JOB_END_TIME': '1781017883', 'SLURM_TASKS_PER_NODE': '4', 'SLURM_NNODES': '1', 'SLURM_GET_USER_ENV': '1', 'SLURM_JOB_START_TIME': '1781017703', 'SLURM_JOB_NODELIST': 'cmmg001', 'SLURM_CLUSTER_NAME': 'cmmc', 'SLURM_NODELIST': 'cmmg001', 'SLURM_NTASKS': '4', 'SLURM_JOB_CPUS_PER_NODE': '8', 'SLURM_TOPOLOGY_ADDR': 'xxx-xxx-xxx.cmmg001', 'SLURM_JOB_NAME': 'executorlib_cache', 'SLURM_JOBID': '20332320', 'SLURM_CONF': '/etc/slurm/slurm.conf', 'SLURM_JOB_QOS': 'normal', 'SLURM_TOPOLOGY_ADDR_PATTERN': 'switch.node', 'SLURM_CPUS_ON_NODE': '8', 'SLURM_JOB_NUM_NODES': '1', 'SLURM_JOB_UID': '35052', 'SLURM_JOB_PARTITION': 's.cmmg', 'SLURM_SCRIPT_CONTEXT': 'prolog_task', 'SLURM_JOB_US

In [7]:
from executorlib import get_cache_data

In [8]:
import subprocess

In [9]:
cache_dir = "executorlib_cache"
for d in get_cache_data(cache_dir):
    if "get_slurm_env" in str(d["function"]):
        print(d['queue_id'])
        shell_output = subprocess.check_output(["sacct", "-j", str(d['queue_id'])], shell=False, universal_newlines=True)

20332320


In [10]:
shell_output.split("\n")

['JobID           JobName  Partition    Account  AllocCPUS      State ExitCode ',
 '------------ ---------- ---------- ---------- ---------- ---------- -------- ',
 '20332320     executorl+     s.cmmg        mpd          8  COMPLETED      0:0 ',
 '20332320.ba+      batch                   mpd          8  COMPLETED      0:0 ',
 '20332320.ex+     extern                   mpd          8  COMPLETED      0:0 ',
 '']

## Submit MPI parallel Python function to SLURM
One of the key features of executorlib is supporting [mpi4py](https://mpi4py.readthedocs.io/en/stable/) parallel Python functions.

In [11]:
def get_slurm_parallel_env():
    import os
    from mpi4py import MPI

    comm = MPI.COMM_WORLD
    size = comm.Get_size()
    rank = comm.Get_rank()

    env_dict = os.environ
    data = {
        k:v for k, v in os.environ.items() if "SLURM_" in k
    }
    data = comm.gather(data, root=0)
    return data

The important part here is the `pmi_mode="pmix"` to use the right backend:

In [12]:
with executorlib.SlurmClusterExecutor(pmi_mode="pmix") as exe:
    f1 = exe.submit(
        get_slurm_parallel_env,
        resource_dict={
            "submission_template": submission_template, 
            "run_time_max": 180,  # in seconds  
            # "partition": "s.cmfe",
            "partition": "s.cmmg",
            "cores": 4,
            "threads_per_core": 4,
        })
    result = f1.result()
    print(result)

/u/janj/projects/executorlib/src/executorlib/task_scheduler/base.py:156: UserWarning: The following keys are not recognized and cannot be validated: ['partition']
  self._validator(resource_dict=resource_dict)


[[{'SLURM_MEM_PER_CPU': '1500', 'SLURM_NODEID': '0', 'SLURM_TASK_PID': '272999', 'SLURM_PRIO_PROCESS': '0', 'SLURM_SUBMIT_DIR': '/cmmc/u/janj/notebooks/2026/2026-06-03-prabhath-exe/executorlib_cache/get_slurm_parallel_env24890a8e967e7328abd85429d1ce0371', 'SLURM_CELL': 'CMMC', 'SLURM_PROCID': '0', 'SLURM_JOB_GID': '12500', 'SLURM_JOB_END_TIME': '1781017894', 'SLURM_TASKS_PER_NODE': '4', 'SLURM_NNODES': '1', 'SLURM_GET_USER_ENV': '1', 'SLURM_JOB_START_TIME': '1781017714', 'SLURM_JOB_NODELIST': 'cmmg001', 'SLURM_CLUSTER_NAME': 'cmmc', 'SLURM_NODELIST': 'cmmg001', 'SLURM_NTASKS': '4', 'SLURM_JOB_CPUS_PER_NODE': '32', 'SLURM_TOPOLOGY_ADDR': 'xxx-xxx-xxx.cmmg001', 'SLURM_JOB_NAME': 'executorlib_cache', 'SLURM_JOBID': '20332321', 'SLURM_CONF': '/etc/slurm/slurm.conf', 'SLURM_JOB_QOS': 'normal', 'SLURM_TOPOLOGY_ADDR_PATTERN': 'switch.node', 'SLURM_CPUS_ON_NODE': '32', 'SLURM_JOB_NUM_NODES': '1', 'SLURM_JOB_UID': '35052', 'SLURM_JOB_PARTITION': 's.cmmg', 'SLURM_SCRIPT_CONTEXT': 'prolog_task', 

In [13]:
for d in get_cache_data(cache_dir):
    if "get_slurm_parallel_env" in str(d["function"]):
        print(d['queue_id'])
        shell_output = subprocess.check_output(["sacct", "-j", str(d['queue_id'])], shell=False, universal_newlines=True)

20332321


In [14]:
shell_output.split("\n")

['JobID           JobName  Partition    Account  AllocCPUS      State ExitCode ',
 '------------ ---------- ---------- ---------- ---------- ---------- -------- ',
 '20332321     executorl+     s.cmmg        mpd         32    RUNNING      0:0 ',
 '20332321.ba+      batch                   mpd         32    RUNNING      0:0 ',
 '20332321.ex+     extern                   mpd         32    RUNNING      0:0 ',
 '20332321.0       python                   mpd         32    RUNNING      0:0 ',
 '']

## Nested Executors
Finally, to gain the most of the flexibility executorlib offers the option to nest executors within each other

In [15]:
def get_nested_env():
    with executorlib.SlurmJobExecutor(max_workers=4, pmi_mode="pmix") as exe:
        future_lst = []
        for i in range(4):
            future_lst.append(exe.submit(
                get_slurm_env,
                resource_dict={"cores": 1, "threads_per_core": 1},
            ))
        return [f.result() for f in future_lst]

In [16]:
from time import sleep

In [17]:
with executorlib.SlurmClusterExecutor(pmi_mode="pmix") as exe:
    f1 = exe.submit(
        get_nested_env,
        resource_dict={
            "submission_template": submission_template, 
            "run_time_max": 180,  # in seconds  
            # "partition": "s.cmfe",
            "partition": "s.cmmg",
            "cores": 1,
            "threads_per_core": 4,
        })

    sleep(10)
    df = get_cache_data(cache_dir)
    for d in df:
        if "get_nested_env" in str(d["function"]):
            print(d['queue_id'])
            shell_output = subprocess.check_output(["sacct", "-j", str(d['queue_id'])], shell=False, universal_newlines=True)
            
    result = f1.result()
    print(result)

/u/janj/projects/executorlib/src/executorlib/task_scheduler/base.py:156: UserWarning: The following keys are not recognized and cannot be validated: ['partition']
  self._validator(resource_dict=resource_dict)


20332322
[{'SLURM_TRES_PER_TASK': 'cpu=1', 'SLURM_MEM_PER_CPU': '1500', 'SLURM_NODEID': '0', 'SLURM_TASK_PID': '273267', 'SLURM_PRIO_PROCESS': '0', 'SLURM_SUBMIT_DIR': '/cmmc/u/janj/notebooks/2026/2026-06-03-prabhath-exe/executorlib_cache/get_nested_envfa45bec89a84ca0b105107cb298c9a0d', 'SLURM_CELL': 'CMMC', 'SLURM_PROCID': '0', 'SLURM_JOB_GID': '12500', 'SLURM_JOB_END_TIME': '1781017904', 'SLURM_TASKS_PER_NODE': '1', 'SLURM_NNODES': '1', 'SLURM_GET_USER_ENV': '1', 'SLURM_JOB_START_TIME': '1781017724', 'SLURM_JOB_NODELIST': 'cmmg001', 'SLURM_CLUSTER_NAME': 'cmmc', 'SLURM_NODELIST': 'cmmg001', 'SLURM_NTASKS': '1', 'SLURM_JOB_CPUS_PER_NODE': '8', 'SLURM_TOPOLOGY_ADDR': 'xxx-xxx-xxx.cmmg001', 'SLURM_JOB_NAME': 'executorlib_cache', 'SLURM_JOBID': '20332322', 'SLURM_CONF': '/etc/slurm/slurm.conf', 'SLURM_JOB_QOS': 'normal', 'SLURM_TOPOLOGY_ADDR_PATTERN': 'switch.node', 'SLURM_CPUS_ON_NODE': '2', 'SLURM_JOB_NUM_NODES': '1', 'SLURM_JOB_UID': '35052', 'SLURM_JOB_PARTITION': 's.cmmg', 'SLURM_SC

In [18]:
shell_output.split("\n")

['JobID           JobName  Partition    Account  AllocCPUS      State ExitCode ',
 '------------ ---------- ---------- ---------- ---------- ---------- -------- ',
 '20332322     executorl+     s.cmmg        mpd          8    RUNNING      0:0 ',
 '20332322.ba+      batch                   mpd          8    RUNNING      0:0 ',
 '20332322.ex+     extern                   mpd          8    RUNNING      0:0 ',
 '20332322.0       python                   mpd          1    RUNNING      0:0 ',
 '20332322.1       python                   mpd          1    RUNNING      0:0 ',
 '20332322.2       python                   mpd          1    RUNNING      0:0 ',
 '20332322.3       python                   mpd          1    RUNNING      0:0 ',
 '']

In [19]:
import pandas

In [20]:
pandas.DataFrame(df)

,function,input_args,input_kwargs,output,resource_dict,runtime,queue_id,error_log_file,filename
0,<function get_slurm_env at 0x14d04fd31260>,[],{},"{'SLURM_MEM_PER_CPU': '1500', 'SLURM_NODEID': ...",{'submission_template': '#!/bin/bash #SBATCH -...,5.015562,20332320,None,/cmmc/u/janj/notebooks/2026/2026-06-03-prabhat...
1,<function get_slurm_parallel_env at 0x14d04fd3...,[],{},"[[{'SLURM_MEM_PER_CPU': '1500', 'SLURM_NODEID'...",{'submission_template': '#!/bin/bash #SBATCH -...,0.011143,20332321,None,/cmmc/u/janj/notebooks/2026/2026-06-03-prabhat...
2,<function get_nested_env at 0x14d04fd311c0>,[],{},NaN,{'submission_template': '#!/bin/bash #SBATCH -...,NaN,20332322,None,/cmmc/u/janj/notebooks/2026/2026-06-03-prabhat...


In [21]:
for d in get_cache_data(cache_dir):
    if "get_nested_env" in str(d["function"]):
        print(d['queue_id'])
        shell_output = subprocess.check_output(["sacct", "-j", str(d['queue_id'])], shell=False, universal_newlines=True)

20332322


In [22]:
shell_output.split("\n")

['JobID           JobName  Partition    Account  AllocCPUS      State ExitCode ',
 '------------ ---------- ---------- ---------- ---------- ---------- -------- ',
 '20332322     executorl+     s.cmmg        mpd          8  COMPLETED      0:0 ',
 '20332322.ba+      batch                   mpd          8  COMPLETED      0:0 ',
 '20332322.ex+     extern                   mpd          8  COMPLETED      0:0 ',
 '20332322.0       python                   mpd          1  COMPLETED      0:0 ',
 '20332322.1       python                   mpd          1  COMPLETED      0:0 ',
 '20332322.2       python                   mpd          1  COMPLETED      0:0 ',
 '20332322.3       python                   mpd          1  COMPLETED      0:0 ',
 '']

## Clean up

In [23]:
import os
import shutil

In [24]:
cache_dir = "executorlib_cache"
os.listdir(cache_dir)

['get_slurm_envccfcb90850bb2803c9b37c6ecde017ee',
 'get_nested_envfa45bec89a84ca0b105107cb298c9a0d_o.h5',
 'get_nested_envfa45bec89a84ca0b105107cb298c9a0d',
 'get_slurm_envccfcb90850bb2803c9b37c6ecde017ee_o.h5',
 'get_slurm_parallel_env24890a8e967e7328abd85429d1ce0371',
 'get_slurm_parallel_env24890a8e967e7328abd85429d1ce0371_o.h5']

In [25]:
shutil.rmtree(cache_dir)